In [47]:
import duckdb
import io
from ollama import chat
import pandas as pd
import numpy as np

In [48]:
rawdata = './data/HomeIntellex_1.csv'
outpath = './data/HomeIntellex_1.parquet'

colnames = ['status', 'date', 'time', 'sensor']
df = pd.read_csv(rawdata, header=None, names=colnames)

df.to_parquet(outpath)

In [49]:
model = 'qwen3:8b'

def get_activities(data, step = 1, previous = pd.DataFrame()):
    print(f'Summarizing step {step}...')
    format = '|Activity|Start Time|End Time|Duration|Notes|'
    system_prompt = f"You are a data scientist tasked with interpreting home sensor data from sensors placed around a subject's house. Provide your answers in the following tabular format for easy parsing: {format}"

    if not previous.empty:
        context = "Use the last timestep for context. {previous}"

    # Chat with a system prompt
    response = chat(model, 
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user', 'content': f'Create a summary of activities within the following csv time window. Include in your summary your best guess as to what might be happening: {data}'}
                    ])
    # We want to grab the table from the output
    table_str = "\n".join([line for line in response.message.content.split('\n') if line.strip().startswith('|')])

    # Read into a DataFrame
    df = pd.read_csv(io.StringIO(table_str), sep="|", skipinitialspace=True).dropna(axis=1, how='all')

    # Clean column names (removing whitespace)
    df.columns = df.columns.str.strip()

    # Remove '---' from rows (part of the way the llm generates tables)
    df = df.replace(r'-{2,}', np.nan, regex=True)
    df = df.dropna()

    print(df)
    # Return the activities summary
    return(df.to_dict('records'))



In [50]:
data_location = './data/HomeIntellex_1.parquet'

db = duckdb.connect()
db.execute(f"CREATE VIEW subject1 AS SELECT * FROM read_parquet('{data_location}')")

# Set timestep boundaries
start = 0
stop = 500
step = 250

# Initialize the activities dictionary
activities = []

query = "SELECT * FROM subject1 LIMIT ? OFFSET ?"

for offset in range(start, stop, step):
    timestep = db.execute(query, [step, offset]).df()
    # Get the last timestep if possible
    if offset !=0:
        previous = timestep = db.execute(query, [step, offset - step]).df()
    else:
        previous=pd.DataFrame()
    activities.extend(get_activities(timestep, round(offset / step) + 1, previous))

db.close()

Summarizing step 1...
                 Activity Start Time  End Time              Duration  \
1           Arriving Home   20:00:03  20:03:02  3 minutes 59 seconds   
2      Leaving Livingroom   20:48:18  20:48:21             3 seconds   
3        Entering Kitchen   20:48:18  20:48:18             0 seconds   
4  Re-entering Livingroom   20:49:07  20:49:08              1 second   
5    Moving Between Rooms   20:48:18  20:49:08              1 minute   

                                               Notes  
1  Multiple entrance sensor activations suggest t...  
2  Short duration indicates quick exit from the l...  
3  Simultaneous with leaving living room, suggest...  
4  Quick re-entry after leaving, indicating brief...  
5  Combined activity of exiting living room, ente...  
Summarizing step 2...
                  Activity Start Time  End Time              Duration  \
1          Arrival at Home   20:00:03  20:03:02  3 minutes 59 seconds   
2       Leaving Livingroom   20:48:18  20:48:21

In [51]:
# Export output to csv
df = pd.DataFrame.from_dict(activities)
print(df)
df.to_csv('./data/output.csv')

                  Activity Start Time  End Time              Duration  \
0            Arriving Home   20:00:03  20:03:02  3 minutes 59 seconds   
1       Leaving Livingroom   20:48:18  20:48:21             3 seconds   
2         Entering Kitchen   20:48:18  20:48:18             0 seconds   
3   Re-entering Livingroom   20:49:07  20:49:08              1 second   
4     Moving Between Rooms   20:48:18  20:49:08              1 minute   
5          Arrival at Home   20:00:03  20:03:02  3 minutes 59 seconds   
6       Leaving Livingroom   20:48:18  20:48:21             3 seconds   
7         Entering Kitchen   20:48:18  20:48:21             3 seconds   
8  Returning to Livingroom   20:48:21  20:49:07            46 seconds   
9   Re-entering Livingroom   20:49:07  20:49:08              1 second   

                                               Notes  
0  Multiple entrance sensor activations suggest t...  
1  Short duration indicates quick exit from the l...  
2  Simultaneous with leaving li